# Phase 2: Benchmark Validation Across Seeds

Goal: confirm the no-adoption, no-forecast baseline behaves as a clean control. Across many seeds the descriptive statistics (mean, std, lag-1 autocorrelation, kurtosis) should cluster tightly around their theoretical values, and the rolling effective AR coefficient should stay flat in expectation.

Reference: section 3 of the proposal for the equations and section 4.2 for the diagnostic statistics. Phase 5 in CLAUDE.md describes this as the no-drift control we need before any erosion claim is meaningful.

In [ ]:
# Number of traders.
N = 128

# Number of periods per seed.
T = 5000

# Market-maker price-impact coefficient (equation 5).
mu = 0.05

# Reduced-form residual AR(1) coefficient in the return law (equation 6).
phi = 0.10

# Standard deviation of exogenous news shocks.
sigma_news = 0.01

# Standard deviation of individual null orders (equation 9).
sigma_q = 1.0

# Number of independent seeds to draw.
num_seeds = 100

# First seed; subsequent seeds are base_seed + 1, base_seed + 2, ...
base_seed = 0

# Window for the rolling effective phi estimate.
rolling_window = 1000

# Output paths.
fig_dir = "../results/figures"
data_dir = "../results/data"

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import matplotlib.pyplot as plt

from reflexive_market import simulate
from reflexive_market.metrics import kurtosis, lag1_autocorr, rolling_phi

In [ ]:
seeds = np.arange(base_seed, base_seed + num_seeds)

mean_per_seed = np.zeros(num_seeds)
std_per_seed = np.zeros(num_seeds)
rho_per_seed = np.zeros(num_seeds)
kurt_per_seed = np.zeros(num_seeds)
phi_traces = np.zeros((num_seeds, T))

for i, seed in enumerate(seeds):
    rng = np.random.default_rng(int(seed))
    out = simulate.run(
        T=T, N=N, mu=mu, phi=phi,
        sigma_news=sigma_news, sigma_q=sigma_q, rng=rng,
    )
    r = out["returns"]
    mean_per_seed[i] = r.mean()
    std_per_seed[i] = r.std()
    rho_per_seed[i] = lag1_autocorr(r)
    kurt_per_seed[i] = kurtosis(r)
    phi_traces[i] = rolling_phi(r, rolling_window)

In [ ]:
print(f"input phi:                  {phi:+.4f}")
print(f"per-seed mean:              {mean_per_seed.mean():+.5f}  std {mean_per_seed.std():.5f}")
print(f"per-seed std:               {std_per_seed.mean():.5f}   std {std_per_seed.std():.5f}")
print(f"per-seed lag-1 autocorr:    {rho_per_seed.mean():+.4f}   std {rho_per_seed.std():.4f}")
print(f"per-seed excess kurtosis:   {kurt_per_seed.mean():+.4f}   std {kurt_per_seed.std():.4f}")

## Per-seed mean return

With Gaussian news and Gaussian null orders the unconditional return mean is zero. The histogram should be centered at zero with a width that shrinks like 1/sqrt(T).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(mean_per_seed, bins=20, alpha=0.7)
ax.axvline(0.0, color="k", linewidth=0.8, linestyle="--", label="theoretical mean = 0")
ax.set_title(f"Per-seed return mean across {num_seeds} seeds")
ax.set_xlabel("Mean return")
ax.set_ylabel("Number of seeds")
ax.legend()
fig.tight_layout()
fig.savefig(f"{fig_dir}/phase_02_seed_mean.png", dpi=150)
plt.show()

## Per-seed return std

Should cluster tightly around its sampling distribution since both inputs to the return are Gaussian and N is large enough for the central limit theorem to bite on aggregate demand.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(std_per_seed, bins=20, alpha=0.7)
ax.set_title(f"Per-seed return std across {num_seeds} seeds")
ax.set_xlabel("Std of returns")
ax.set_ylabel("Number of seeds")
fig.tight_layout()
fig.savefig(f"{fig_dir}/phase_02_seed_std.png", dpi=150)
plt.show()

## Per-seed lag-1 autocorrelation

The empirical lag-1 autocorrelation should cluster around the input phi. Spread is sampling noise of order 1/sqrt(T).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(rho_per_seed, bins=20, alpha=0.7)
ax.axvline(phi, color="k", linewidth=0.8, linestyle="--", label=f"input phi = {phi}")
ax.set_title("Per-seed lag-1 autocorrelation")
ax.set_xlabel("Lag-1 autocorrelation")
ax.set_ylabel("Number of seeds")
ax.legend()
fig.tight_layout()
fig.savefig(f"{fig_dir}/phase_02_seed_lag1.png", dpi=150)
plt.show()

## Per-seed excess kurtosis

If the return distribution is approximately Gaussian under the null rule, excess kurtosis should sit near zero across seeds.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(kurt_per_seed, bins=20, alpha=0.7)
ax.axvline(0.0, color="k", linewidth=0.8, linestyle="--", label="Gaussian = 0")
ax.set_title("Per-seed excess kurtosis")
ax.set_xlabel("Excess kurtosis")
ax.set_ylabel("Number of seeds")
ax.legend()
fig.tight_layout()
fig.savefig(f"{fig_dir}/phase_02_seed_kurtosis.png", dpi=150)
plt.show()

## Rolling effective phi

Each faint line is one seed's rolling lag-1 autocorrelation. The thick line is the mean across seeds, which should sit near the input phi and not drift over time. This is the no-adoption, no-erosion control: any future drift in later phases is therefore mechanism, not artifact.

In [ ]:
mean_phi_trace = np.full(T, np.nan)
std_phi_trace = np.full(T, np.nan)
valid_from = rolling_window - 1
mean_phi_trace[valid_from:] = phi_traces[:, valid_from:].mean(axis=0)
std_phi_trace[valid_from:] = phi_traces[:, valid_from:].std(axis=0)

fig, ax = plt.subplots(figsize=(8, 4))
for i in range(num_seeds):
    ax.plot(phi_traces[i], color="C0", alpha=0.05, linewidth=0.5)
ax.plot(mean_phi_trace, color="black", linewidth=1.5, label="mean across seeds")
ax.axhline(phi, color="C3", linewidth=1.0, linestyle="--", label=f"input phi = {phi}")
ax.set_title(f"Rolling lag-1 autocorrelation (window = {rolling_window})")
ax.set_xlabel("Time period")
ax.set_ylabel("Rolling phi estimate")
ax.legend()
fig.tight_layout()
fig.savefig(f"{fig_dir}/phase_02_rolling_phi.png", dpi=150)
plt.show()

## Save numeric summary

Per-seed scalars plus the across-seed mean and std of the rolling phi trace. The full per-seed traces are not saved because they balloon the file without adding diagnostic value beyond the mean and std bands.

In [ ]:
np.savez(
    f"{data_dir}/phase_02_benchmark.npz",
    mean_per_seed=mean_per_seed,
    std_per_seed=std_per_seed,
    rho_per_seed=rho_per_seed,
    kurt_per_seed=kurt_per_seed,
    mean_phi_trace=mean_phi_trace,
    std_phi_trace=std_phi_trace,
    phi_input=np.array(phi),
    rolling_window=np.array(rolling_window),
    seeds=seeds,
)

## Done when

- Per-seed mean clusters tightly around zero.
- Per-seed std clusters around its sampling distribution with no outliers.
- Per-seed lag-1 autocorrelation clusters near the input phi.
- Per-seed excess kurtosis sits near zero.
- The mean rolling phi trace is flat at the input phi after the warm-up window.